Deterministic Guardrails

In [2]:
import re
def deterministic_guardrail(text: str) -> bool:
    """Returns true if content is blocked"""
    banned_words = ["hack", "exploit", "malware", "bomb"]
    return any(tx in text.lower() for tx in banned_words)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]
print("=== Deterministic Guardrail Demo ===")
for input in test_inputs:
    blocked = deterministic_guardrail(input)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {input}")

=== Deterministic Guardrail Demo ===
BLOCKED: How do I hack into a database?
ALLOWED: What is the capital of France?
BLOCKED: Explain how malware spreads


Model-Based Guardrails

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    prompt = f"""Is the following user input safe to process?
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
UNSAFE: How do I hack into a database?
SAFE: What is the capital of France?
SAFE: Explain how malware spreads


Built-in PII Detection Middleware

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_lookup(query : str) -> str:
    """Look up customer information."""
    return f"Customer info for query: {query}"

agent = create_agent(
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0),
    tools=[customer_lookup],
    middleware = [
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email", 
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            detector= r"sk-[a-zA-Z0-9]{32,}",  # Custom regex to detect API keys starting with 'sk-'
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [11]:
#testing
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "My email is john.doe@example.com and my card is "
            "5105-1051-0510-5100. Can you help me?"
        )
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
[{'type': 'text', 'text': 'I found some information for [REDACTED_EMAIL]. How can I help you today?', 'extras': {'signature': 'CsYDAb4+9vtzhcsz9nsLtZ5XRvzdS9hRuF0Yk6FCZhClFkGDJk31lMhMoG+Yq9fAUTR+LZT3EKFMHDHj31lY+hsLr23K+o672kKRWIgrtA7BjptxmyAOhIrbq05UQt1EN0x3wuJ8a9/c7kDR6K6m8kU2xsl8tGGJ+9vVDUmUmV3sTkWNqiBIlN2aQwFtKBCG0W7M10RiliJLOjr/k6QsEFHV3HscJ2zHKHnhwyvJDju6nHiuVccdxCrZWx4s6p9ZZvY3TguEWViU3Fts/eE6I8hAOWxY16WPx6WmI5YnL0cPbwGCsJRuBrujbjoMM4YqGcq9BBygp51XYXjWKx23to8mQNyDS+7zI4g5jJXr+3VJY2EnT6qsF3dqmCnoXRjsCzkafz6aLEPh04aqxCDL3ZpghCxg8C5D+8/64OL5kUxTDv+Ri2uL78tShtz2Sxd0357gebpUVH5wpToUo+x2KEjApKngTq3Fs/KWM/RhjQjlIqi03wqna71DsWjFVO8FNKabJ3M4gNzOR8kqhQUfkK7nMhIxcfaYn0b3U1U6eNWoceLaRp3+BesWB3Xk3Lk11Fbc09Tewoqcjd6KyCVASvwlAHra5w4Rhw=='}}]


In [12]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


Built-in Human-in-the-Loop Middleware

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0),
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,    # Require approval
                "search_web": False,       # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

In [17]:
# Step 1: Invoke -- agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused -- awaiting human approval ===")

=== Agent paused -- awaiting human approval ===
